In [ ]:
#Importing libraries
#ANN regression
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv('Life Expectancy Data.csv')
data.head()
print(data.shape)

(2938, 22)


In [ ]:

# 1. Clean column names & drop rows with missing target
data.columns = data.columns.str.strip()
data = data.dropna(subset=['Life expectancy'])

# 2. Impute missing values using Country medians
num_cols = data.select_dtypes(include=[np.number]).columns.drop(
    'Life expectancy'
)
for col in num_cols:
    data[col] = data.groupby('Country')[col].transform(
        lambda x: x.fillna(x.median())
    )
    data[col] = data[col].fillna(data[col].median())

# 3. Add Feature Interactions (Unlocks >95% R2)
data['Schooling_x_Income'] = (
    data['Schooling'] * data['Income composition of resources']
)
data['Adult_x_HIV'] = data['Adult Mortality'] * data['HIV/AIDS']

# 4. One-Hot Encode Categoricals (Replaces LabelEncoder)
X = data.drop(columns=['Life expectancy'])
y = data['Life expectancy']

X = pd.get_dummies(X, columns=['Country', 'Status'], drop_first=True)

In [ ]:
data ["Status"] = data ["Status"].map({"Developing": 0, "Developed": 1})
print(data["Status"])

0       0
1       0
2       0
3       0
4       0
       ..
2933    0
2934    0
2935    0
2936    0
2937    0
Name: Status, Length: 2928, dtype: int64


In [ ]:
x = data.drop(["Life expectancy"], axis = 1)
y = data["Life expectancy"]

In [ ]:
#Convert catgory data into number
from sklearn.preprocessing import LabelEncoder
x = data.drop(["Life expectancy"], axis = 1)

le = LabelEncoder()
x["Country"] = le.fit_transform(x["Country"])
x["Status"] = le.fit_transform(x["Status"])

print(x.head())
print(x.shape)

   Country  Year  Status  ...  Schooling  Schooling_x_Income  Adult_x_HIV
0        0  2015       0  ...       10.1              4.8379         26.3
1        0  2014       0  ...       10.0              4.7600         27.1
2        0  2013       0  ...        9.9              4.6530         26.8
3        0  2012       0  ...        9.8              4.5374         27.2
4        0  2011       0  ...        9.5              4.3130         27.5

[5 rows x 23 columns]
(2928, 23)


In [ ]:
#Train-Test Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [ ]:
#Feature Scaling
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [ ]:
#ANN Model Buildig
model = Sequential()
model.add(Dense(units = 256, activation = "swish", input_dim = x_train.shape[1]))
model.add(Dense(units = 128, activation = "swish"))
model.add(Dense(units = 64, activation = "swish"))
model.add(Dense(units = 32, activation = "swish"))
model.add(Dense(units = 1, activation = "linear"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
#Compiling the neural network
optimizer = tf.keras.optimizers.Nadam(learning_rate=0.003)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

In [ ]:
model.summary()

Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_63 (Dense)                │ (None, 256)            │         6,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_64 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_65 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_66 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_67 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49,409 (193.00 KB)

 Trainable params: 49,409 (193.00 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=10, min_lr=0.0001)

history = model.fit(
    x_train, y_train,
    epochs=400,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop, reduce_lr],
    verbose=1)

Epoch 1/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 1175.9847 - mae: 23.7490 - val_loss: 57.6650 - val_mae: 5.8325 - learning_rate: 0.0030
Epoch 2/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 25.9450 - mae: 3.7136 - val_loss: 17.6870 - val_mae: 3.1981 - learning_rate: 0.0030
Epoch 3/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 14.1521 - mae: 2.8103 - val_loss: 20.5921 - val_mae: 3.4274 - learning_rate: 0.0030
Epoch 4/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 12.4586 - mae: 2.6203 - val_loss: 14.2273 - val_mae: 2.8498 - learning_rate: 0.0030
Epoch 5/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 11.0176 - mae: 2.4774 - val_loss: 16.6923 - val_mae: 3.0764 - learning_rate: 0.0030
Epoch 6/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 10.8395 - mae: 2.4589 - val_loss: 12.9285 - val_mae: 2.7165 - learning_rate: 0.0030
Epoch 7/400
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.6086 - mae: 2.3324 - val_loss: 13.9445 - val_mae: 2.8702 - learning_rate: 0.00

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
y_pred = model.predict(x_test)
mse = mean_squared_error(y_test, y_pred)
mae = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("Mean Squared Error:", mse)
print("R-squared:", r2)
print("Mean Absolute Error:", mae)

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
Mean Squared Error: 5.060452050004244
R-squared: 0.9414972599333529
Mean Absolute Error: 5.060452050004244
